Import Library

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

Library digunakan untuk membantu proses analisis data dan pembuatan model machine learning. *pandas* digunakan untuk membaca dataset berbentuk tabel. *train_test_split* digunakan untuk membagi data training dan validasi. *StandardScaler* digunakan untuk normalisasi data agar skala fitur seimbang. Beberapa model klasifikasi digunakan untuk membandingkan performa model dan memilih model terbaik. *accuracy_score* dan *confusion_matrix* digunakan untuk mengevaluasi performa model.

# Persiapan Data

Membaca Dataset

In [2]:
train = pd.read_csv('data_training.csv')
test = pd.read_csv('data_testing.csv')

Dataset training dan testing dibaca menggunakan *pandas*. Dataset training digunakan untuk melatih model karena memiliki variabel target *quality*, sedangkan dataset testing digunakan untuk melakukan prediksi kualitas wine.

Menampilkan Dataset

In [3]:
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.3,0.740,0.08,1.7,0.094,10.0,45.0,0.99576,3.24,0.50,9.8,5,1366
1,8.1,0.575,0.22,2.1,0.077,12.0,65.0,0.99670,3.29,0.51,9.2,5,103
2,10.1,0.430,0.40,2.6,0.092,13.0,52.0,0.99834,3.22,0.64,10.0,7,942
3,12.9,0.500,0.55,2.8,0.072,7.0,24.0,1.00012,3.09,0.68,10.9,6,811
4,8.4,0.360,0.32,2.2,0.081,32.0,79.0,0.99640,3.30,0.72,11.0,6,918
...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,6.7,1.040,0.08,2.3,0.067,19.0,32.0,0.99648,3.52,0.57,11.0,4,1467
853,8.0,0.390,0.30,1.9,0.074,32.0,84.0,0.99717,3.39,0.61,9.0,5,1533
854,7.4,0.350,0.33,2.4,0.068,9.0,26.0,0.99470,3.36,0.60,11.9,6,1580
855,7.9,0.570,0.31,2.0,0.079,10.0,79.0,0.99677,3.29,0.69,9.5,6,1216


Dataset ditampilkan untuk melihat struktur data, nama kolom, serta memastikan data berhasil dibaca dengan benar.
Langkah ini membantu memahami fitur-fitur yang akan digunakan dalam proses klasifikasi.

# Pembersihan Data

Cek Missing Value

In [4]:
train.isnull().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


In [5]:
test.isnull().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


Pengecekan missing value dilakukan untuk memastikan tidak terdapat data kosong pada dataset.
Data yang memiliki missing value dapat mempengaruhi performa model machine learning sehingga perlu dicek sebelum proses pelatihan model dilakukan. Berdasarkan hasil pengecekan, tidak terdapat missing value pada dataset.

Memisahkan Fitur dan Target

In [6]:
X = train.drop('quality', axis=1)
y = train['quality']

Variabel *quality* dijadikan sebagai target prediksi (y) karena merupakan nilai yang ingin diprediksi oleh model.
Sedangkan seluruh variabel selain *quality* digunakan sebagai fitur (X) yang akan membantu model mempelajari pola data wine.

Split Data

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Dataset dibagi menjadi data training dan data validasi.
Sebanyak 80% data digunakan untuk melatih model dan 20% digunakan untuk pengujian model.
*random_state=42* digunakan agar pembagian data konsisten setiap kali program dijalankan.

Feature Scaling

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

Feature scaling dilakukan menggunakan *StandardScaler* agar seluruh fitur memiliki skala yang seimbang.
Normalisasi data penting karena beberapa fitur memiliki rentang nilai yang berbeda sehingga dapat mempengaruhi proses pembelajaran model.

# Perbandingan dan Pembuatan Model

Membandingkan Beberapa Model

In [9]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
}

In [10]:
hasil_akurasi = []

for nama, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    hasil_akurasi.append([nama, accuracy])

    print(nama)
    print("Accuracy :", accuracy)
    print("-" * 30)

Decision Tree
Accuracy : 0.5058139534883721
------------------------------
KNN
Accuracy : 0.5
------------------------------
SVM
Accuracy : 0.5465116279069767
------------------------------
Naive Bayes
Accuracy : 0.47674418604651164
------------------------------
Random Forest
Accuracy : 0.6046511627906976
------------------------------


In [11]:
akurasi_df = pd.DataFrame(
    hasil_akurasi,
    columns=['Model', 'Accuracy']
)

akurasi_df

,Model,Accuracy
0,Decision Tree,0.505814
1,KNN,0.500000
2,SVM,0.546512
3,Naive Bayes,0.476744
4,Random Forest,0.604651


Beberapa model klasifikasi dibandingkan untuk mengetahui model dengan performa terbaik berdasarkan nilai accuracy. Model dengan nilai accuracy tertinggi dipilih untuk digunakan pada proses prediksi data testing.

Membuat dan Melatih Model Terbaik (Random Forest)

In [12]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

Model Random Forest dipilih karena menghasilkan performa yang baik dalam klasifikasi kualitas wine. Model kemudian dilatih menggunakan data training. Random Forest bekerja dengan menggabungkan banyak decision tree untuk meningkatkan akurasi prediksi dan mengurangi overfitting. Parameter *n_estimators=200* menunjukkan bahwa model menggunakan 200 decision tree. Fungsi *fit()* digunakan untuk melatih model menggunakan data training.

# Evaluasi Model

Evaluasi Model

In [13]:
y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print("Accuracy Score:", accuracy)

Accuracy Score: 0.6046511627906976


Model yang telah dilatih digunakan untuk memprediksi data validasi.
Nilai akurasi dihitung menggunakan *accuracy_score* untuk mengetahui tingkat ketepatan model dalam memprediksi kualitas wine. Semakin tinggi nilai akurasi, maka semakin baik performa model. Model Random Forest menghasilkan akurasi yang cukup baik dalam memprediksi kualitas wine.

In [14]:
cm = confusion_matrix(y_val, y_pred)

print(cm)

[[ 0  1  2  0  0]
 [ 0 47 20  0  0]
 [ 0 23 49  6  0]
 [ 0  2 11  8  0]
 [ 0  0  0  3  0]]


Confusion matrix digunakan untuk melihat perbandingan antara hasil prediksi model dengan data aktual.
Melalui confusion matrix dapat diketahui jumlah prediksi yang benar maupun salah pada setiap kelas kualitas wine. Berdasarkan confusion matrix, model Random Forest sudah mampu memprediksi sebagian besar data dengan benar yang terlihat dari nilai diagonal confusion matrix. Namun, masih terdapat beberapa kesalahan prediksi pada kelas kualitas yang memiliki karakteristik mirip. Secara keseluruhan, model memiliki performa yang cukup baik dalam melakukan klasifikasi kualitas wine.

# Prediksi Data Uji

Prediksi Data Testing

In [15]:
X_test = test.copy()

In [16]:
X_test = scaler.transform(X_test)

In [17]:
test_pred = model.predict(X_test)

Dataset testing dipersiapkan untuk dilakukan prediksi menggunakan model yang telah dilatih.
Data testing juga dilakukan scaling menggunakan scaler yang sama agar skala data konsisten dengan data training.
Selanjutnya model digunakan untuk memprediksi nilai kualitas wine pada data testing.

Membuat Hasil Prediksi

In [18]:
hasil = pd.DataFrame({
    'Id': test['Id'],
    'Quality': test_pred
})

Hasil prediksi disimpan ke dalam DataFrame baru yang hanya berisi kolom *Id* dan *quality* sesuai format yang diminta pada instruksi.

In [19]:
hasil

,Id,Quality
0,222,5
1,1514,5
2,417,5
3,754,5
4,516,5
...,...,...
281,1147,6
282,296,5
283,170,5
284,1439,5


Hasil prediksi ditampilkan untuk memastikan data prediksi berhasil dibuat dengan benar dan sesuai format.

Menyimpan File CSV

In [20]:
hasil.to_csv(
    'hasilprediksi_043.csv',
    index=False,
    sep=';'
)

Hasil prediksi disimpan dalam format CSV menggunakan fungsi *to_csv()*.
Parameter *index=False* digunakan agar index DataFrame tidak ikut tersimpan ke dalam file CSV.
Parameter *sep=';'* digunakan agar file CSV dapat terbaca dengan rapi pada Microsoft Excel.